# Airspace Object Detection: MBAOD Model + Cross-Dataset Test

This notebook trains on **Clean-MBAOD**, the Roboflow copy of version 1: 35,106 images in six classes (airplane, bird, helicopter, kite, parachute, uav). It then tests that model on **AOD 4** images it has never seen. That cross-dataset test is experiment 3 in the plan.

**Setup:** same as the week 1 notebook. Set **GPU T4 x2**, turn **Internet** on, and attach the `ROBOFLOW_API_KEY` secret.
Run this **after** the AOD 4 baseline finishes, so the two sessions don't share GPU quota.

**Reference point:** the MBAOD authors report **mAP50 of 98%** with the much larger YOLOv11-L. Expect a few points lower from YOLO11s.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# 1. Download Clean-MBAOD (v1) and AOD 4 (v6) in YOLO format
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

mbaod = rf.workspace("fmk-mwmyi").project("mbaod").version(1).download("yolov8", location="/kaggle/working/mbaod")
aod4  = rf.workspace("airborne-object-detection").project("airborne-object-detection-4-aod4").version(6).download("yolov8", location="/kaggle/working/aod4")

In [ ]:
# 2. Class names, split sizes and boxes per class for MBAOD
import yaml, os, glob
from collections import Counter

mb_yaml = os.path.join(mbaod.location, "data.yaml")
mb_names = yaml.safe_load(open(mb_yaml))["names"]
print("MBAOD classes:", mb_names)
for split in ["train", "valid", "test"]:
    print(f"{split}: {len(glob.glob(os.path.join(mbaod.location, split, 'images', '*')))} images")

counts = Counter()
for f in glob.glob(os.path.join(mbaod.location, "train", "labels", "*.txt")):
    for line in open(f):
        if line.strip():
            counts[int(line.split()[0])] += 1
for i, n in enumerate(mb_names):
    print(f"{n:12s} {counts[i]}")

In [ ]:
# 3. Train YOLO11s on MBAOD (same settings as the AOD 4 baseline, so results compare fairly)
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data=mb_yaml, epochs=50, imgsz=640, batch=32, device=[0, 1],
            project="/kaggle/working/runs", name="mbaod_640", patience=15)

In [ ]:
# 4. Evaluate on MBAOD's own test split
best = YOLO("/kaggle/working/runs/mbaod_640/weights/best.pt")
m = best.val(data=mb_yaml, split="test", imgsz=640, plots=True,
             project="/kaggle/working/runs", name="mbaod_test")
print(f"MBAOD test  mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
for i, n in enumerate(mb_names):
    print(f"{n:12s} AP50-95 = {m.box.maps[i]:.3f}")

from IPython.display import Image, display
display(Image("/kaggle/working/runs/mbaod_test/confusion_matrix_normalized.png", width=700))

## Cross-dataset test: MBAOD model on AOD 4 images

AOD 4 calls drones `drone` and MBAOD calls them `uav`. The cell below copies the AOD 4 test set and rewrites its label IDs to MBAOD's class order. The model can then be scored on it directly. Kite and parachute simply have no examples in this set.

In [ ]:
# 5. Remap AOD 4 test labels into MBAOD class IDs
import shutil
aod_names = yaml.safe_load(open(os.path.join(aod4.location, "data.yaml")))["names"]
rename = {"drone": "uav"}
id_map = {i: mb_names.index(rename.get(n, n)) for i, n in enumerate(aod_names)}
print("AOD 4 -> MBAOD id map:", {aod_names[k]: mb_names[v] for k, v in id_map.items()})

dst = "/kaggle/working/aod4_as_mbaod/test"
os.makedirs(dst + "/labels", exist_ok=True)
if not os.path.exists(dst + "/images"):
    shutil.copytree(os.path.join(aod4.location, "test", "images"), dst + "/images")
for f in glob.glob(os.path.join(aod4.location, "test", "labels", "*.txt")):
    out = []
    for line in open(f):
        p = line.split()
        if p:
            out.append(" ".join([str(id_map[int(p[0])])] + p[1:]))
    open(os.path.join(dst, "labels", os.path.basename(f)), "w").write("\n".join(out))

cross_yaml = "/kaggle/working/aod4_as_mbaod/data.yaml"
yaml.safe_dump({"path": "/kaggle/working/aod4_as_mbaod", "train": "test/images",
                "val": "test/images", "test": "test/images", "names": mb_names}, open(cross_yaml, "w"))

In [ ]:
# 6. Score the MBAOD model on AOD 4 test images it has never seen
c = best.val(data=cross_yaml, split="test", imgsz=640, plots=True,
             project="/kaggle/working/runs", name="mbaod_on_aod4")
print(f"MBAOD model on AOD 4  mAP50: {c.box.map50:.3f}   mAP50-95: {c.box.map:.3f}")
for n in ["airplane", "bird", "helicopter", "uav"]:
    print(f"{n:12s} AP50-95 = {c.box.maps[mb_names.index(n)]:.3f}")
display(Image("/kaggle/working/runs/mbaod_on_aod4/confusion_matrix_normalized.png", width=700))

## Write down

| Model | Tested on | mAP50 | mAP50-95 |
|---|---|---|---|
| YOLO11s trained on AOD 4 | AOD 4 test | (from week 1 notebook) | |
| YOLO11s trained on MBAOD | MBAOD test | | |
| YOLO11s trained on MBAOD | **AOD 4 test (unseen)** | | |

The gap between rows 2 and 3 is your **generalization drop**, the headline of experiment 3. Also note which class drops most. My guess is drone/uav, because the two datasets' drones look very different.